# Run the whole pipeline (steps 01 to 10)
Runs every step **from its own file**, in order, in one session, so this notebook never holds a second copy of any SQL. It is the table of contents for the lab, and the shape a production DAG takes (see `docs/PRODUCTION.md`).

- **Scope:** whatever `lab_settings` holds. Set it first in `00.set_session_variables.ipynb` (run that notebook so the scope is saved).
- **Two gates** (in the *Settings and helpers* cell) decide whether this run may download data or spend AI credits. A gated step that is off is reported as `SKIPPED`, never silently left out.
- **Always runs, regardless of gates (no cost):** 01, 04, 06, 09, 10.
- **Every query is tagged** `lab:<step file>` (Snowflake `QUERY_TAG`), so query history and AI cost can be grouped by step.
- **Ends with:** a step summary, `pipeline_coverage` (one row per year in scope) and the step 10 data-quality checks.

### Standing check (do this before every class and after any code edit)
Run All with **both gates `False`**. It is free and verifies the whole pipeline wiring: expect no `FAILED` step and *all* step 10 checks passing.

### One source per step
To change what a step does, edit **that step's file** (`sql/NN...sql`, the 08 notebook, or `python/NN...py`), then run it here. Never paste step SQL into this notebook (a unit test enforces this).

### Gate quick reference

| Situation | `RUN_INGESTION` | `RUN_AI_STEPS` |
| --- | --- | --- |
| Standing check / see where every year stands | `False` | `False` |
| New years or a new ticker in scope: load them only | `True` | `False` |
| PDFs + SEC facts already loaded, pages/extraction still to do | `False` | `True` |
| New years or ticker, run everything end to end | `True` | `True` |
| Fresh 08 comparison of reports already extracted | `False` | `True` + set `only_new_reports = FALSE` in the 08 notebook |
| After any run | set both back to `False` | |

**Cost note:** 05 and 07 never repeat work. 08 starts a new batch each run but, with its default `only_new_reports = TRUE`, only extracts reports that have not been extracted with the same questions (`prompt_version`). Changing the questions or `include_all_metrics` re-extracts automatically; `only_new_reports = FALSE` re-extracts everything deliberately.

In [ ]:
import importlib.util
import sys
import time
from pathlib import Path

import pandas as pd
from snowflake.snowpark.context import get_active_session

# =================================================================================
# GATES: the only two settings in this notebook. Set them deliberately, every run.
# =================================================================================
# SUMMARY
#   A gate decides whether THIS run is allowed to do something that downloads data
#   or costs AI credits. It is not about what ran before. Steps 01, 04, 06, 09, 10
#   always run (free). A gated step that is off prints "SKIPPED" with the reason and
#   appears in the step summary, so nothing is silently left out.
#   Default and resting state: both False. Set them back to False after every run.
#   Standing check: Run All with both False before every class / after any edit.
#
# RUN_INGESTION  -> steps 02 (PDFs) and 03 (SEC facts). Needs internet access
#                   (PUBLIC_FILING_ACCESS integration on this notebook's service).
#   True  when the scope in 00 includes years or a ticker not loaded yet.
#   False when the PDFs and SEC facts for the scope are already loaded (the usual case).
#   Cost: no AI credits. Re-downloads every PDF in scope and reloads SEC facts:
#         safe (overwrite / MERGE) but slow if nothing changed.
#   If off and data is missing: pipeline_coverage shows next_step
#         '02 ingest PDF' or '03 ingest SEC facts' for that year.
#
# RUN_AI_STEPS   -> steps 05 (locate pages: AI_PARSE_DOCUMENT), 07 (first AI_EXTRACT)
#                   and 08 (both extraction paths). Costs Snowflake AI credits; check
#                   current rates in the Snowflake docs (AI function costs) first.
#   True  when any year still needs pages located or extraction (coverage next_step
#         is '05 ...' or '08 ...'), or when you deliberately want a fresh 08 batch.
#   False to check status, rerun evaluation (09) / validation (10), or after a run.
#   What re-runs cost:
#     05  only parses reports not yet approved, never the same page twice (~21 pages each).
#     07  only extracts reports it has not extracted before.
#     08  new batch each run, but by default (only_new_reports = TRUE in the 08 notebook)
#         only extracts reports not yet extracted with the same questions
#         (prompt_version). Set only_new_reports = FALSE there for a deliberate full
#         re-extraction of every READY report in scope.
#
# COMMON COMBINATIONS
#   Free status check ............................ RUN_INGESTION=False, RUN_AI_STEPS=False
#   Load new years only .......................... True,  False
#   Data loaded, finish locate + extract ......... False, True
#   New years/ticker, everything end to end ...... True,  True
#
# ATP NOTE: RUN_INGESTION - change to True only when the scope includes new years or
#   a new ticker. It re-downloads every PDF in scope and reloads the SEC facts, which is
#   safe but slow if nothing changed.
# ATP NOTE: RUN_AI_STEPS - change to True when you want 05, 07 and 08 to run.
# ---------------------------------------------------------------------------------
RUN_INGESTION = False   # steps 02-03: download PDFs and SEC facts (internet)
RUN_AI_STEPS = False    # steps 05, 07, 08: AI_PARSE_DOCUMENT / AI_EXTRACT (credits)
# ---------------------------------------------------------------------------------

session = get_active_session()
for statement in ("USE ROLE all_users_role", "USE DATABASE bootcamp",
                  "USE WAREHOUSE compute_wh", "USE SCHEMA atp524"):
    session.sql(statement).collect()

# Project folders (works whether the kernel starts in the project root or python/).
cwd = Path.cwd().resolve()
ROOT = next(d for d in (cwd, *cwd.parents) if (d / "sql").is_dir() and (d / "python").is_dir())
if str(ROOT / "python") not in sys.path:
    sys.path.insert(0, str(ROOT / "python"))
from sql_runner import run_sql_file  # noqa: E402  (needs the path above)

def load_module(name, filename):
    """Numbered filenames cannot be imported by name; load them by path."""
    spec = importlib.util.spec_from_file_location(name, ROOT / "python" / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

STEP_LOG = []  # one row per step: what ran, what was skipped, how long it took

def run_step(step, path_or_callable, uses_ai=False, uses_internet=False, show_last=True):
    """Run one pipeline step and record its outcome; never skip without saying so."""
    if uses_ai and not RUN_AI_STEPS:
        STEP_LOG.append((step, "SKIPPED (AI) - set RUN_AI_STEPS = True", 0.0))
        print(f"[{step}] SKIPPED: calls AI functions; set RUN_AI_STEPS = True to run it")
        return None
    if uses_internet and not RUN_INGESTION:
        STEP_LOG.append((step, "SKIPPED (ingestion) - set RUN_INGESTION = True", 0.0))
        print(f"[{step}] SKIPPED: downloads data; set RUN_INGESTION = True to run it")
        return None
    started = time.time()
    try:
        if callable(path_or_callable):
            result = path_or_callable(session)
            print(f"[{step}] {result}")
        else:
            result = run_sql_file(session, ROOT / path_or_callable, show=False)
            print(f"[{step}] ran {len(result)} statements from {path_or_callable}")
    except Exception:
        STEP_LOG.append((step, "FAILED", round(time.time() - started, 1)))
        raise
    STEP_LOG.append((step, "OK", round(time.time() - started, 1)))
    if show_last and isinstance(result, list) and result and result[-1][1]:
        return pd.DataFrame([row.as_dict() for row in result[-1][1]])
    return None

print(f"Project root: {ROOT}\nRUN_INGESTION={RUN_INGESTION}  RUN_AI_STEPS={RUN_AI_STEPS}")

In [ ]:
%%sql -r scope
-- Set in 00.set_session_variables.ipynb. Every step below processes every year in it.
SELECT setting_id, ticker, first_year, last_year, updated_at FROM lab_settings;

## 01 Create tables, keys and documentation
Creates every table (grouped by layer: CONFIG, RAW, CURATED), declares keys and grains, and adds table comments. Safe to re-run.

In [ ]:
run_step("01 ddls", "sql/01.ddls.sql")

## 02-03 Ingest the scope's PDFs and SEC facts
Both read the scope from `lab_settings`. Any year that does not arrive stays visible in `pipeline_coverage`.

In [ ]:
pdfs = load_module("ingest_annual_report_pdfs", "02.ingest_annual_report_pdfs.py")
run_step("02 ingest PDFs", pdfs.main, uses_internet=True)

In [ ]:
sec = load_module("ingest_sec_ground_truth", "03.ingest_sec_ground_truth.py")
run_step("03 ingest SEC facts", sec.main, uses_internet=True)

## 04-06 Prepare, locate pages, check readiness
04 proposes each report's period and accession; 05 parses ~21 pages per new report to find the statements and approve it (AI); 06 builds `report_readiness` and `pipeline_coverage`.

In [ ]:
run_step("04 prepare manifest", "sql/04.prepare_manifest.sql")

In [ ]:
run_step("05 locate statement pages", "sql/05.locate_statement_pages.sql", uses_ai=True)

In [ ]:
run_step("06 check readiness", "sql/06.check_readiness.sql", show_last=False)
session.table("pipeline_coverage").select("report_year", "readiness", "next_step").sort("report_year").to_pandas()

## 07-08 Extract (AI)
07 is the quick three-question extraction per READY report; 08 runs both extraction paths for every READY report as one batch (its SQL is read from the 08 notebook's SQL cells).

In [ ]:
run_step("07 first extraction", "sql/07.first_extraction.sql", uses_ai=True)

In [ ]:
run_step("08 extract reports", "sql/08.extract_reports.ipynb", uses_ai=True)

## 09-10 Evaluate and validate
09 scores the latest batch against SEC ground truth; 10 runs the key, grain, relationship and accepted-value checks.

In [ ]:
run_step("09 evaluate results", "sql/09.evaluate_results.sql")

In [ ]:
validation = run_sql_file(session, ROOT / "sql/10.validation.sql", show=False)
STEP_LOG.append(("10 validation", "OK", 0.0))
# The key/grain suite is the result set with a CHECK_TYPE column; failures sort first.
suite = next(rows for _, rows in validation if rows and "CHECK_TYPE" in rows[0].as_dict())
checks = pd.DataFrame([r.as_dict() for r in suite])
failed = checks[checks.STATUS != "PASS"]
print(f"{len(checks) - len(failed)} of {len(checks)} data-quality checks pass")
if len(failed):
    STEP_LOG[-1] = ("10 validation", f"FAILED: {len(failed)} check(s)", 0.0)
checks

## Summary
What ran, what was skipped (and why), and where every year in scope stands.

In [ ]:
pd.DataFrame(STEP_LOG, columns=["step", "outcome", "seconds"])

In [ ]:
%%sql -r coverage
-- One row per year in scope; next_step names the first step still missing.
SELECT ticker, report_year, pdf_on_stage, sec_facts_loaded, period_proposed, approved,
       readiness, direct_pdf_done, parse_then_extract_done, next_step
FROM pipeline_coverage
ORDER BY report_year;